In [1]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def report_memory(name):
    if torch.cuda.is_available():
        print(f"{name} - Allocated: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")
    else:
        return

In [4]:
class CheckpointedBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        print("=> Checkpointedblock initialized")
    
    def forward(self, x):
        output = self.block(x)
        return output

In [5]:
class MyModel(nn.Module):
    def __init__(self, use_checkpointing=True):
        super().__init__()
        self.use_checkpointing = use_checkpointing
        self.input_layer = nn.Linear(10, 20)

        # heavy block
        self.heavy_block = CheckpointedBlock(20, 65536, 30)

        self.output_layer = nn.Linear(30, 5)
    
    def forward(self, x):
        x = self.input_layer(x)
        if self.use_checkpointing:
            x = checkpoint(self.heavy_block, x, use_reentrant=False)
        else:
            x = self.heavy_block(x)
        x = self.output_layer(x)
        return x

In [6]:
report_memory("init")

init - Allocated: 0.00 MB


## Block1, Checkpointing

In [8]:
model_A = MyModel(use_checkpointing=True)
model_A.to(device)

dummy_input = torch.randn(8192, 10, device=device, requires_grad=True)
report_memory("A - before forward")

output_A = model_A(dummy_input)
report_memory("A - after forward")

object_fn = output_A.sum()
object_fn.backward()
report_memory("A - after backward")

=> Checkpointedblock initialized
A - before forward - Allocated: 56.60 MB
A - after forward - Allocated: 58.17 MB
A - after backward - Allocated: 44.54 MB


## Block2, No Checkpointing

In [9]:
model_B = MyModel(use_checkpointing=False)
model_B.to(device)

dummy_input = torch.randn(8192, 10, device=device, requires_grad=True)
report_memory("B - before forward")

output_B = model_B(dummy_input)
report_memory("B - after forward")

object_fn = output_B.sum()
object_fn.backward()
report_memory("B - after backward")

=> Checkpointedblock initialized
B - before forward - Allocated: 57.60 MB
B - after forward - Allocated: 2107.32 MB
B - after backward - Allocated: 70.83 MB


In [10]:
print(x.grad)

None


In [11]:
y.sum().backward()

In [14]:
x.grad

tensor([ 3.7955, -0.1956, -2.6305,  ..., -1.4922,  1.3648,  0.7376])

In [17]:
storage[0]

tensor([ 1.8977, -0.0978, -1.3152,  ..., -0.7461,  0.6824,  0.3688],
       requires_grad=True)